# 04 — Deploy the React Dashboard

This notebook deploys the HydraB Vehicle 360 dashboard as an SPCS service.
The Docker image is pre-built — you just create a service pointing to it.

After this notebook, you'll have a **public URL** for your personal dashboard.

In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA GOLD;
USE WAREHOUSE HYDRAB_HOL_WH;

In [ ]:
-- Create the SPCS service from pre-built image
CREATE SERVICE IF NOT EXISTS DASHBOARD_SERVICE
  IN COMPUTE POOL HYDRAB_HOL_POOL
  FROM SPECIFICATION $$
  spec:
    containers:
    - name: dashboard
      image: /HYDRAB_HOL_SHARED/PUBLIC/IMAGE_REPO/hydrab-dashboard:v1
      env:
        SNOWFLAKE_DATABASE: HYDRAB_HOL_NHUVAERE
      resources:
        requests:
          cpu: 0.5
          memory: 512M
        limits:
          cpu: 1
          memory: 1G
    endpoints:
    - name: dashboard
      port: 3000
      public: true
  $$
  MIN_INSTANCES = 1
  MAX_INSTANCES = 1;

In [ ]:
-- Wait for service to start (check status)
-- Run this cell a few times until status = RUNNING
SELECT SYSTEM$GET_SERVICE_STATUS('GOLD.DASHBOARD_SERVICE');

In [ ]:
-- Get the public endpoint URL
SHOW ENDPOINTS IN SERVICE GOLD.DASHBOARD_SERVICE;

## Dashboard Deployed!

Copy the `ingress_url` from above and open it in your browser.
You should see the HydraB Vehicle 360 dashboard with:
- Overview stats (vehicles, pipeline, deliveries)
- Fleet Map with GPS pins
- Sales Pipeline funnel chart
- Vehicle telemetry detail pages

---

## What's Next: Extend with Cortex Code Desktop

The `react-app/` folder in this skill contains the full source code.
Open it in Cortex Code Desktop and ask CoCo to:
- Add a new page (e.g. Delivery Tracking)
- Connect charts to live Snowflake queries
- Improve the Fleet Map with vehicle popups
- Add the Cortex Agent chat to the dashboard

In [ ]:
-- Cleanup (run when done with the lab)
-- DROP SERVICE IF EXISTS GOLD.DASHBOARD_SERVICE;
-- DROP DATABASE IF EXISTS IDENTIFIER($user_db);